# Traffic Data Analysis - Indonesian Cities

This notebook provides interactive analysis of traffic data for **Semarang**, **Bandung**, and **Jakarta**.

**Data Summary:**
- Date Range: March 2025 - February 2026
- Time Periods: 8 periods per day
- Segments: Semarang (1,076), Bandung (3,069), Jakarta (14,549)

## 1. Setup and Import Libraries

In [ ]:
# Import required libraries
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Libraries imported successfully!')

In [ ]:
# Define constants
CITIES = {
    'smg': {'name': 'Semarang', 'folder': 'traffic_smg_output', 'color': '#2ecc71'},
    'bdg': {'name': 'Bandung', 'folder': 'traffic_bdg_output', 'color': '#3498db'},
    'jkt': {'name': 'Jakarta', 'folder': 'traffic_jkt_output', 'color': '#e74c3c'}
}

TIME_PERIODS = [
    'night', 'morning_peak', 'morning_offpeak', 'lunch_hours',
    'afternoon_offpeak', 'evening_peak', 'evening_offpeak', 'late_night'
]

TIME_LABELS = {
    'night': '00-06',
    'morning_peak': '06-09',
    'morning_offpeak': '09-12',
    'lunch_hours': '12-14',
    'afternoon_offpeak': '14-16',
    'evening_peak': '16-19',
    'evening_offpeak': '19-22',
    'late_night': '22-00'
}

print('Constants defined!')

## 2. Load and Explore Data

In [ ]:
# Load evening peak data for all cities
smg = gpd.read_file('traffic_smg_output/evening_peak_smg.gpkg')
bdg = gpd.read_file('traffic_bdg_output/evening_peak_bdg.gpkg')
jkt = gpd.read_file('traffic_jkt_output/evening_peak_jkt.gpkg')

print(f"Semarang: {len(smg)} segments")
print(f"Bandung:  {len(bdg)} segments")
print(f"Jakarta:  {len(jkt)} segments")

In [ ]:
# View data structure
print("Columns:", list(jkt.columns))
print("\nData types:")
print(jkt.dtypes)

In [ ]:
# Preview Jakarta data
jkt.head(10)

In [ ]:
# Statistical summary
jkt[['jam_factor_mean', 'jam_factor_std', 'jam_factor_count', 'jam_factor_min', 'jam_factor_max']].describe()

## 3. Compare Cities

In [ ]:
# Compare basic statistics
comparison = pd.DataFrame({
    'City': ['Semarang', 'Bandung', 'Jakarta'],
    'Segments': [len(smg), len(bdg), len(jkt)],
    'Mean Jam Factor': [smg['jam_factor_mean'].mean(), bdg['jam_factor_mean'].mean(), jkt['jam_factor_mean'].mean()],
    'Std': [smg['jam_factor_mean'].std(), bdg['jam_factor_mean'].std(), jkt['jam_factor_mean'].std()],
    'Min': [smg['jam_factor_mean'].min(), bdg['jam_factor_mean'].min(), jkt['jam_factor_mean'].min()],
    'Max': [smg['jam_factor_mean'].max(), bdg['jam_factor_mean'].max(), jkt['jam_factor_mean'].max()]
}).round(3)

comparison

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 6))

cities = ['Semarang', 'Bandung', 'Jakarta']
means = [smg['jam_factor_mean'].mean(), bdg['jam_factor_mean'].mean(), jkt['jam_factor_mean'].mean()]
colors = ['#2ecc71', '#3498db', '#e74c3c']

bars = ax.bar(cities, means, color=colors, edgecolor='black', linewidth=1.2)
ax.axhline(y=1.0, color='green', linestyle='--', alpha=0.7, label='Free Flow (1.0)')
ax.axhline(y=2.0, color='orange', linestyle='--', alpha=0.7, label='Moderate (2.0)')

ax.set_ylabel('Mean Jam Factor', fontsize=12)
ax.set_title('Evening Peak Traffic Comparison', fontsize=14, fontweight='bold')
ax.legend()

# Add value labels
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, 
            f'{mean:.2f}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Traffic Maps

In [ ]:
# Plot traffic maps for all cities
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (gdf, name, color) in zip(axes, [(smg, 'Semarang', '#2ecc71'), 
                                          (bdg, 'Bandung', '#3498db'), 
                                          (jkt, 'Jakarta', '#e74c3c')]):
    gdf.plot(column='jam_factor_mean', cmap='RdYlGn_r', linewidth=0.5, 
             ax=ax, vmin=0, vmax=3, legend=True,
             legend_kwds={'label': 'Jam Factor', 'shrink': 0.8})
    ax.set_title(f'{name} - Evening Peak', fontsize=14, fontweight='bold')
    ax.set_axis_off()

plt.tight_layout()
plt.show()

## 5. Time Period Analysis

In [ ]:
# Load all time periods for each city
def load_all_periods(city_code):
    folder = CITIES[city_code]['folder']
    data = {}
    for period in TIME_PERIODS:
        data[period] = gpd.read_file(f'{folder}/{period}_{city_code}.gpkg')
    return data

smg_data = load_all_periods('smg')
bdg_data = load_all_periods('bdg')
jkt_data = load_all_periods('jkt')

print('All time periods loaded!')

In [ ]:
# Calculate mean jam factor for each period
def get_period_means(city_data):
    return [city_data[p]['jam_factor_mean'].mean() for p in TIME_PERIODS]

smg_means = get_period_means(smg_data)
bdg_means = get_period_means(bdg_data)
jkt_means = get_period_means(jkt_data)

# Create DataFrame
daily_pattern = pd.DataFrame({
    'Period': TIME_PERIODS,
    'Time': [TIME_LABELS[p] for p in TIME_PERIODS],
    'Semarang': smg_means,
    'Bandung': bdg_means,
    'Jakarta': jkt_means
}).round(3)

daily_pattern

In [ ]:
# Plot daily traffic pattern
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(TIME_PERIODS))
width = 0.25

ax.bar(x - width, smg_means, width, label='Semarang', color='#2ecc71')
ax.bar(x, bdg_means, width, label='Bandung', color='#3498db')
ax.bar(x + width, jkt_means, width, label='Jakarta', color='#e74c3c')

ax.axhline(y=1.0, color='green', linestyle='--', alpha=0.5)
ax.axhline(y=2.0, color='orange', linestyle='--', alpha=0.5)

ax.set_xlabel('Time Period', fontsize=12)
ax.set_ylabel('Mean Jam Factor', fontsize=12)
ax.set_title('Daily Traffic Pattern Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f"{p}\n({TIME_LABELS[p]})" for p in TIME_PERIODS], fontsize=9)
ax.legend(title='City')
ax.set_ylim(0, 2.5)

plt.tight_layout()
plt.show()

In [ ]:
# Line plot version
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(TIME_PERIODS, smg_means, 'o-', label='Semarang', color='#2ecc71', linewidth=2, markersize=8)
ax.plot(TIME_PERIODS, bdg_means, 's-', label='Bandung', color='#3498db', linewidth=2, markersize=8)
ax.plot(TIME_PERIODS, jkt_means, '^-', label='Jakarta', color='#e74c3c', linewidth=2, markersize=8)

ax.fill_between(TIME_PERIODS, 0, 1, alpha=0.1, color='green', label='Free Flow Zone')
ax.fill_between(TIME_PERIODS, 1, 2, alpha=0.1, color='yellow')
ax.fill_between(TIME_PERIODS, 2, 3, alpha=0.1, color='red')

ax.set_xlabel('Time Period', fontsize=12)
ax.set_ylabel('Mean Jam Factor', fontsize=12)
ax.set_title('Daily Traffic Rhythm', fontsize=14, fontweight='bold')
ax.legend(loc='upper left')
ax.set_ylim(0, 2.5)
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 6. Congestion Distribution Analysis

In [ ]:
# Distribution of jam factors
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (gdf, name, color) in zip(axes, [(smg, 'Semarang', '#2ecc71'), 
                                          (bdg, 'Bandung', '#3498db'), 
                                          (jkt, 'Jakarta', '#e74c3c')]):
    ax.hist(gdf['jam_factor_mean'], bins=30, color=color, alpha=0.7, edgecolor='white')
    ax.axvline(gdf['jam_factor_mean'].mean(), color='red', linestyle='--', 
               label=f"Mean: {gdf['jam_factor_mean'].mean():.2f}")
    ax.axvline(gdf['jam_factor_mean'].median(), color='blue', linestyle='--',
               label=f"Median: {gdf['jam_factor_mean'].median():.2f}")
    ax.set_xlabel('Jam Factor', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title(f'{name} - Evening Peak', fontsize=12, fontweight='bold')
    ax.legend()

plt.suptitle('Distribution of Traffic Congestion', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot comparison
fig, ax = plt.subplots(figsize=(10, 6))

data_to_plot = [smg['jam_factor_mean'], bdg['jam_factor_mean'], jkt['jam_factor_mean']]
bp = ax.boxplot(data_to_plot, labels=['Semarang', 'Bandung', 'Jakarta'], patch_artist=True)

colors = ['#2ecc71', '#3498db', '#e74c3c']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Jam Factor', fontsize=12)
ax.set_title('Traffic Congestion Distribution (Evening Peak)', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Identify Congested Segments

In [ ]:
# Find most congested segments in Jakarta
def classify_congestion(gdf):
    gdf = gdf.copy()
    conditions = [
        gdf['jam_factor_mean'] <= 1.0,
        (gdf['jam_factor_mean'] > 1.0) & (gdf['jam_factor_mean'] <= 2.0),
        (gdf['jam_factor_mean'] > 2.0) & (gdf['jam_factor_mean'] <= 4.0),
        gdf['jam_factor_mean'] > 4.0
    ]
    categories = ['Free Flow', 'Light Traffic', 'Moderate', 'Heavy']
    gdf['congestion_level'] = np.select(conditions, categories, default='Unknown')
    return gdf

jkt_classified = classify_congestion(jkt)

# Count by category
congestion_counts = jkt_classified['congestion_level'].value_counts()
print("Jakarta Evening Peak - Congestion Classification:")
print(congestion_counts)
print(f"\nPercentages:")
print((congestion_counts / len(jkt_classified) * 100).round(1))

In [ ]:
# Top 10 most congested segments
top_congested = jkt.nlargest(10, 'jam_factor_mean')[['fid', 'jam_factor_mean', 'jam_factor_max', 'jam_factor_count']]
print("Top 10 Most Congested Segments in Jakarta (Evening Peak):")
top_congested

In [ ]:
# Visualize congestion classification
fig, ax = plt.subplots(figsize=(12, 10))

colors_map = {'Free Flow': '#27ae60', 'Light Traffic': '#f1c40f', 'Moderate': '#e67e22', 'Heavy': '#c0392b'}

for level in ['Free Flow', 'Light Traffic', 'Moderate', 'Heavy']:
    subset = jkt_classified[jkt_classified['congestion_level'] == level]
    if len(subset) > 0:
        subset.plot(ax=ax, color=colors_map[level], linewidth=0.5, label=f"{level} ({len(subset)})")

ax.set_title('Jakarta - Congestion Classification (Evening Peak)', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', title='Congestion Level')
ax.set_axis_off()

plt.tight_layout()
plt.show()

## 8. Statistical Analysis

In [ ]:
# Correlation between morning and evening peak
print("Correlation: Morning Peak vs Evening Peak")
print("="*50)

for city_code, city_data, name in [('smg', smg_data, 'Semarang'), 
                                    ('bdg', bdg_data, 'Bandung'), 
                                    ('jkt', jkt_data, 'Jakarta')]:
    morning = city_data['morning_peak']['jam_factor_mean']
    evening = city_data['evening_peak']['jam_factor_mean']
    
    corr, pval = stats.pearsonr(morning, evening)
    print(f"{name}: r = {corr:.3f}, p-value = {pval:.2e}")

In [ ]:
# T-test: Is Jakarta significantly more congested than Bandung?
t_stat, p_val = stats.ttest_ind(jkt['jam_factor_mean'], bdg['jam_factor_mean'])
print(f"T-test: Jakarta vs Bandung (Evening Peak)")
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value: {p_val:.2e}")
print(f"\nConclusion: {'Significant difference' if p_val < 0.05 else 'No significant difference'} (α=0.05)")

In [ ]:
# ANOVA: Compare all three cities
f_stat, p_val = stats.f_oneway(smg['jam_factor_mean'], bdg['jam_factor_mean'], jkt['jam_factor_mean'])
print(f"One-way ANOVA: All Cities (Evening Peak)")
print(f"F-statistic: {f_stat:.3f}")
print(f"p-value: {p_val:.2e}")
print(f"\nConclusion: {'Significant difference between cities' if p_val < 0.05 else 'No significant difference'}")

## 9. Peak vs Off-Peak Analysis

In [ ]:
# Calculate peak and off-peak averages per segment
def calculate_peak_offpeak(city_data):
    peak_periods = ['morning_peak', 'evening_peak']
    offpeak_periods = ['night', 'morning_offpeak', 'afternoon_offpeak', 'late_night']
    
    peak_values = np.mean([city_data[p]['jam_factor_mean'].values for p in peak_periods], axis=0)
    offpeak_values = np.mean([city_data[p]['jam_factor_mean'].values for p in offpeak_periods], axis=0)
    
    return peak_values, offpeak_values

smg_peak, smg_offpeak = calculate_peak_offpeak(smg_data)
bdg_peak, bdg_offpeak = calculate_peak_offpeak(bdg_data)
jkt_peak, jkt_offpeak = calculate_peak_offpeak(jkt_data)

In [ ]:
# Scatter plot: Peak vs Off-Peak
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (peak, offpeak, name, color) in zip(axes, 
    [(smg_peak, smg_offpeak, 'Semarang', '#2ecc71'),
     (bdg_peak, bdg_offpeak, 'Bandung', '#3498db'),
     (jkt_peak, jkt_offpeak, 'Jakarta', '#e74c3c')]):
    
    ax.scatter(offpeak, peak, alpha=0.3, s=10, color=color)
    
    # Diagonal line
    max_val = max(peak.max(), offpeak.max())
    ax.plot([0, max_val], [0, max_val], 'k--', alpha=0.5)
    
    # Correlation
    corr = np.corrcoef(offpeak, peak)[0, 1]
    
    ax.set_xlabel('Off-Peak Jam Factor', fontsize=11)
    ax.set_ylabel('Peak Jam Factor', fontsize=11)
    ax.set_title(f'{name}\n(r = {corr:.3f})', fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3)

plt.suptitle('Peak vs Off-Peak Traffic Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 10. Export Results

In [ ]:
# Export summary to CSV
summary_df = pd.DataFrame({
    'City': [],
    'Period': [],
    'Mean': [],
    'Std': [],
    'Min': [],
    'Max': []
})

for city_code, city_data in [('Semarang', smg_data), ('Bandung', bdg_data), ('Jakarta', jkt_data)]:
    for period in TIME_PERIODS:
        gdf = city_data[period]
        summary_df = pd.concat([summary_df, pd.DataFrame({
            'City': [city_code],
            'Period': [period],
            'Mean': [gdf['jam_factor_mean'].mean()],
            'Std': [gdf['jam_factor_mean'].std()],
            'Min': [gdf['jam_factor_mean'].min()],
            'Max': [gdf['jam_factor_mean'].max()]
        })], ignore_index=True)

summary_df = summary_df.round(4)
summary_df.to_csv('traffic_summary.csv', index=False)
print("Saved: traffic_summary.csv")
summary_df.head(10)

In [ ]:
# Export Jakarta evening peak to GeoJSON (for web mapping)
jkt_export = jkt.copy()
jkt_export.to_file('jakarta_evening_peak.geojson', driver='GeoJSON')
print("Saved: jakarta_evening_peak.geojson")

## 11. Interactive Map (Optional - requires folium)

In [ ]:
# Install folium if needed
# !pip install folium

try:
    import folium
    from folium import plugins
    
    # Create interactive map for Jakarta
    jkt_wgs84 = jkt.to_crs(epsg=4326)
    center = [jkt_wgs84.geometry.centroid.y.mean(), jkt_wgs84.geometry.centroid.x.mean()]
    
    m = folium.Map(location=center, zoom_start=11, tiles='cartodbpositron')
    
    # Add traffic layer
    folium.GeoJson(
        jkt_wgs84,
        style_function=lambda x: {
            'color': 'red' if x['properties']['jam_factor_mean'] > 2 else 
                     'orange' if x['properties']['jam_factor_mean'] > 1 else 'green',
            'weight': 2,
            'opacity': 0.7
        },
        tooltip=folium.GeoJsonTooltip(fields=['fid', 'jam_factor_mean'],
                                       aliases=['Segment ID:', 'Jam Factor:'])
    ).add_to(m)
    
    # Save map
    m.save('jakarta_traffic_map.html')
    print("Saved: jakarta_traffic_map.html")
    
    # Display in notebook
    m
    
except ImportError:
    print("Folium not installed. Run: pip install folium")

---

## Summary

This notebook demonstrated:

1. **Data Loading** - Loading GeoPackage files with geopandas
2. **City Comparison** - Comparing traffic across Semarang, Bandung, Jakarta
3. **Traffic Maps** - Visualizing spatial patterns
4. **Time Period Analysis** - Understanding daily traffic rhythms
5. **Distribution Analysis** - Histograms and boxplots
6. **Congestion Classification** - Identifying problem areas
7. **Statistical Tests** - T-tests and ANOVA
8. **Peak vs Off-Peak** - Correlation analysis
9. **Data Export** - CSV, GeoJSON outputs
10. **Interactive Maps** - Folium web maps

**Key Findings:**
- Jakarta has the highest congestion levels
- Evening peak (16:00-19:00) is the most congested period
- Night hours show free-flow conditions across all cities